MNIST 

28x28 images of handwritten digits dataset. We want a vector as the example input. So, the inupt matrix would be [batch, image]

We use .view to flatten the image into a 784 sized vector. The input matrix is of size [batch, 784] (mental note about flattening: we don't need to worry about losing dimensionality or information as all examples are flattened in the same manner. Internally, python processes them as a flat array anways (I think)).

In [7]:
import gzip
import torch
import struct
import numpy as np
import random 

def load_images(path):
    with gzip.open(path, 'rb') as f:
        magic, n, rows, cols = struct.unpack('>IIII', f.read(16))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(n, rows, cols)

def load_labels(path):
    with gzip.open(path, 'rb') as f:
        magic, n = struct.unpack('>II', f.read(8))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data


X_train = load_images('gzip/emnist-digits-train-images-idx3-ubyte.gz')
Y_train = load_labels('gzip/emnist-digits-train-labels-idx1-ubyte.gz')

X_test = load_images('gzip/emnist-digits-test-images-idx3-ubyte.gz')
Y_test = load_labels('gzip/emnist-digits-test-labels-idx1-ubyte.gz')

X_train = torch.tensor(X_train)
Y_train = torch.tensor(Y_train)

X_test = torch.tensor(X_test)
Y_test = torch.tensor(Y_test)

print(X_train.shape)
print(Y_train.shape)

print(X_test.shape)
print(Y_test.shape)

torch.Size([240000, 28, 28])
torch.Size([240000])
torch.Size([40000, 28, 28])
torch.Size([40000])


In [3]:
class Node():
    def __init__(self, value, parents=()):
        self.value = value
        self.grad = 0.0
        self.parents = set(parents)
        self.backward = lambda: None

    def __repr__(self):
        return ("Value(data={self.data})")
    
    def __add__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        output = Node(self.value + other.value, (self, other))

        def backward():
            self.grad += output.grad
            other.grad += output.grad
        output.backward = backward
        return output 

    def __mul__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        output = Node(self.value * other.value, (self, other))

        def backward():
            self.grad += output.grad * other.value
            other.grad += output.grad * self.value
        output.backward = backward
        return output
    
    def tanh(self):
        output = Node(np.tanh(self.value), (self,))

        def backward():
            self.grad += output.grad * (1.0 - output.value**2)
        output.backward = backward
        return output
    
    def call_backward(self):
        # Phase 1: Build topological order
        topological = []
        visited = set()

        def build_topo(node):
            if node not in visited:
                visited.add(node)
                for parent in node.parents:
                    build_topo(parent)
                topological.append(node)

        build_topo(self)

        # Phase 2: Iterate in reverse, calling backward
        self.grad = 1.0
        for node in reversed(topological):
            node.backward()

In [17]:
# Testing Node class 
# Manual Forward Pass 
x1 = Node(2.5)
w1 = Node(1.0)

x2 = Node(3.0) 
w2 = Node(-2.0)

a1 = x1 * w1
a2 = x2 * w2

b = a1 + a2

c = b.tanh()

# Call the Backward Pass
c.call_backward()

print(c.grad)
print(b.grad)
print(a1.grad)
print(a2.grad)
print(w1.grad)
print(w2.grad)
print(x1.grad)
print(x2.grad)


1.0
0.003640884720487403
0.003640884720487403
0.003640884720487403
0.009102211801218507
0.010922654161462209
0.003640884720487403
-0.007281769440974806


At this point we have the Node class setup, we have the manual forward pass and the autobackward pass. These operations operate on the individual scalar values so if we have vector and matrix operations we need to apply these elementwise to carry out the bigger operations. In the case of EMNIST the vectorization of the input image is a [28,28] vector which we flatten by representing in a 1D vector that's [784]. This would be 784 neurons in the input layer where each neuron represents the feeatures for all examples being passed to the hidden layer. We want to apply the weights to the features and not examples, we are training on features NOT examples. 

The weights are parameters that determine the relationship between the inputs and the expected outcomes. The architecture determines the family of relationships the model is capable of expressing. The weights determine how much the features contribute to the error where as if we had the examples as neurons we would determine how much a specific example determines the error but that doesn't generalize. Given some set of weights, they determine how much a feature should contribute to the final output, in a simple example of predicting house prices, we have features (sqft, beds, zip), given some training data the weights contributing to the beds might not contribute as much as the weight for the sqft for the price. 

In [11]:
class Neuron:
    def __init__(self, nin):
        self.weights = [Node(torch.randn(1).item() * (1/(nin)**0.5)) for _ in range(nin)] #Xavier/Glorot scaling 1/sqrt(nin) for smart initialization
        self.bias = Node(0)

    def activation(self, inputs):
        dotprod = sum((w * x for w, x in zip(self.weights, inputs)), self.bias)
        return dotprod.tanh()
    

class Layer:
    def __init__(self, nin, non):
        self.neurons = [Neuron(nin) for _ in range(non)]

    def forward(self, inputs):
        return None
        

class MLP:
    def __init__(self, nin, layers):
        layer_list = [nin] + layers
        for x, y in zip(layer_list, layer_list[1:]):
            Layer(x, y)

    def call_forward(self, input_matrix):
        return None




In [18]:
# No longer using Node object to define Neuron, Layer, and MLP
class Neuron:
    def __init__(self, nin):
        self.W = torch.randn(nin) * (1/nin**0.5)
        self.b = torch.zeros(1)
        self.W.requires_grad = True
        self.b.requires_grad = True

    def activation(self, inputs):
        return (self.W @ inputs + self.b).tanh()

class Layer:
    def __init__(self, nin, non):
        self.neurons = [Neuron(nin) for _ in range(non)]

    def forward(self, inputs):
        return torch.stack([n.activation(inputs) for n in self.neurons])

class MLP:
    def __init__(self, nin, layers):
        sizes = [nin] + layers
        self.layers = [Layer(sizes[i], sizes[i+1]) for i in range(len(layers))]

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

In [ ]:
lr = 0.1
batch_size = 28
epochs = 10

W1 = torch.randn(784, 128) * (1/784**0.5)
b1 = torch.zeros(128)
W2 = torch.randn(128, 10) * (1/128**0.5)
b2 = torch.zeros(10)

def softmax(Z):
    exp = torch.exp(Z)
    return exp / torch.sum(exp, dim=1, keepdim=True)

for epoch in range(epochs):
    indices = torch.randperm(len(X_train))
    epoch_loss = 0.0

    for i in range(0, len(X_train), batch_size):
        batch_idx = indices[i:i+batch_size]
        X = X_train[batch_idx].float().reshape(-1, 784) / 255.0           # normalize and reshape from [batch, 28, 28] to [batch, 784]
        Y_idx = Y_train[batch_idx].long()
        Y = torch.zeros(X.shape[0], 10)
        Y[torch.arange(X.shape[0]), Y_idx] = 1                            # one-hot [batch, 10]

        # Forward pass
        Z1 = X @ W1 + b1                           # [batch, 728] x [728, 128] = [batch, 128]
        A1 = torch.tanh(Z1)                        # [batch, 128]
        Z2 = A1 @ W2 + b2                          # [batch, 128] x [128, 10] = [batch, 10]
        A2 = softmax(Z2)                           # [batch, 10]

        # Partial derivatives
        dLdWdA2dZ2 = (A2 - Y) / batch_size         # dL/dZ2  [batch, 10]
        dZ2dW2 = A1                                # [batch, 128]
        dZ2dA1 = W2                                # [128, 10]
        dA1dZ1 = 1 - torch.tanh(Z1)**2             # tanh derivative [batch, 128]
        dZ1dW1 = X                                 # [batch, 784]

        # Chain rule - layer 2
        dLdW2 = dZ2dW2.T @ dLdWdA2dZ2             # [batch, 128]^T x [batch 10] = [128, batch] x [batch, 10] = [128, 10]
        dLdb2 = torch.sum(dLdWdA2dZ2, dim=0)      # [10]

        # Chain rule - layer 1
        dLdA1 = dLdWdA2dZ2 @ dZ2dA1.T             # [batch, 10] x [128, 10]^T = [batch, 10] x [10, 128] = [batch, 128]
        dLdZ1 = dLdA1 * dA1dZ1                    # elementwise [batch, 128]
        dLdW1 = dZ1dW1.T @ dLdZ1                  # [batch, 728]^T x [batch, 128] = [728, batch] x [batch, 128] = [784, 128]
        dLdb1 = torch.sum(dLdZ1, dim=0)           # [128]

        # Update
        W1 -= lr * dLdW1
        b1 -= lr * dLdb1
        W2 -= lr * dLdW2
        b2 -= lr * dLdb2

        epoch_loss += -torch.mean(torch.sum(Y * torch.log(A2 + 1e-8), dim=1)).item()

    print(f"Epoch {epoch+1}, Loss: {epoch_loss / (len(X_train) / batch_size):.4f}")

Epoch 1, Loss: 0.1588
Epoch 2, Loss: 0.0701
Epoch 3, Loss: 0.0517
Epoch 4, Loss: 0.0426
Epoch 5, Loss: 0.0365
Epoch 6, Loss: 0.0321
Epoch 7, Loss: 0.0287
Epoch 8, Loss: 0.0256
Epoch 9, Loss: 0.0229
Epoch 10, Loss: 0.0211


# Summary of Topics & Discoveries

## 1. Node Class (Scalar Autograd)
- Built a `Node` class that tracks value, gradient, and parent nodes
- Implemented `__add__`, `__mul__`, `tanh` with backward functions
- `call_backward` does topological sort then reverse-iterates calling each node's backward
- `__mul__` handles negative values naturally -- no need to overload negation

## 2. Neural Network Architecture Fundamentals
- **Input layer neurons = features, not examples.** The batch dimension is implicit, not part of the architecture
- Neurons represent the **feature space** (columns of the input matrix), examples flow through as rows
- Why features and not examples: weights encode "what to look for in data" -- must generalize to unseen examples. Weights tied to specific examples can't generalize (house price analogy)
- **Weights** = tunable parameters defining the specific relationship between features and outcomes
- **Architecture** (depth, width, activations) = defines the family of relationships the model can express
- **Nonlinearities** (tanh, ReLU) are what make depth matter -- multiple linear layers collapse into one equivalent linear transformation
- Deeper layers operate on increasingly abstract representations; middle layers are often not human-interpretable (mechanistic interpretability is an open field)

## 3. Weight Initialization
- Random weights break symmetry so neurons learn different features
- Bias initialized to 0 -- neutral starting point, no symmetry problem
- `uniform(-1,1)` fine for toy networks
- Xavier/Glorot: `1/sqrt(nin)` -- keeps variance stable for tanh
- He initialization: `sqrt(2/nin)` -- designed for ReLU
- Too large -> tanh saturates, gradients vanish. Too small -> signal dies

## 4. Building Neuron -> Layer -> MLP
- `Neuron`: list of weight Nodes + bias, computes dot product + activation
- `Layer`: list of Neurons, each takes the same input list
- `MLP`: list of Layers, chains them sequentially
- Forward pass: input list -> each neuron produces one scalar -> list of outputs -> next layer's input
- The "matrix multiply" emerges from the structure without ever building an explicit matrix

## 5. Scalar Node Limitation
- Node operates on scalars -- can't batch. One example at a time
- Building a tensor class on top of Node = reinventing PyTorch (weeks of work, extremely slow in pure Python)
- Solution: keep Node for understanding autograd, switch to PyTorch tensors for actual training

## 6. Standard Neural Network Notation
- `A^[0]` = input (X), `W^[l]` = weights, `b^[l]` = bias
- `Z^[l]` = pre-activation (before nonlinearity), `A^[l]` = post-activation
- Edges in diagrams = W, nodes = A, Z lives conceptually between them

## 7. Forward Pass (EMNIST)
- `Z1 = X @ W1 + b1`, `A1 = tanh(Z1)` -- hidden layer
- `Z2 = A1 @ W2 + b2`, `A2 = softmax(Z2)` -- output layer (no tanh on output)
- Softmax needs `axis=1, keepdims=True` to normalize per row, not entire matrix
- Normalize inputs to [0,1] by dividing by 255 -- prevents tanh saturation

## 8. Backward Pass (Manual Derivation)
- Chain rule expansions for dL/dW1, dL/db1, dL/dW2, dL/db2
- Softmax + cross-entropy derivative combines into `dZ2 = (A2 - Y) / m`
- tanh derivative: `1 - tanh^2(Z)`
- **Transpose rule**: for `Z = A @ W`, gradients are `dL/dW = A.T @ dL/dZ` and `dL/dA = dL/dZ @ W.T` -- transposes make shapes match the parameter being updated
- Loss is the bridge between forward and backward -- not part of the network but the starting point of backprop

## 9. Cross-Entropy vs MSE
- MSE has vanishing gradients with softmax -- confidently wrong predictions get tiny gradients
- Cross-entropy cancels the softmax derivative, giving strong gradients even when confidently wrong
- Cross-entropy measures probability on correct class, not geometric distance

## 10. Training Loop
- Shuffle indices each epoch with `torch.randperm`, iterate through all data in batches
- Every example seen exactly once per epoch, different random order each epoch
- Batch size is a computational/optimization choice, not about unique coverage
- Smaller batch = noisier gradient (can help escape local minima), larger = smoother
- Loss must be accumulated across all batches for meaningful per-epoch reporting

## 11. Generalization & Distribution
- Good predictions limited to "within distribution"
- Models don't know when their knowledge doesn't apply -- OOD detection is an open problem
- LLMs answer confidently because training objective rewards confident text
- RLHF patches the behavior (performing uncertainty) without solving the underlying problem

## 12. Train / Dev / Test Split
- **Training**: forward + backward + update
- **Dev/validation**: forward pass only, compute loss for monitoring -- no backward, no weight updates
- **Test**: forward pass only, once at the end, unbiased evaluation
- Computing loss is a forward operation -- `.backward()` computes gradients for updates, not loss